# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Build a chat model client for any OpenAI-compatible endpoint.
2. Define a **tool** — a plain Python function — with the `@tool` decorator.
3. Create an agent with `create_agent` and run it.
4. Stream the agent's response token-by-token.

## Setup

Prerequisites: run `pip install -r requirements.txt` in the repository root, copy `.env.example` to `.env`, fill in `LLM_BASE_URL`, `LLM_API_KEY`, `LLM_MODEL`, and run `python scripts/check_endpoint.py`.

The cell below loads those variables from `.env` and builds the chat model client. `ChatOpenAI` speaks the OpenAI Chat Completions protocol, so the same code works with DeepSeek, OpenAI, a local Ollama server, or any other compatible endpoint — only the three environment variables change. `LLM_EXTRA_BODY` carries optional provider-specific request options (for DeepSeek it turns thinking mode off).

In [1]:
import os
from dotenv import find_dotenv, load_dotenv
%cd C:\Users\21808\pyt-aiae202-template
print("当前工作目录:", os.getcwd())
env_path = find_dotenv()
print("找到的.env完整路径:", env_path)

if env_path:
    load_dotenv(env_path)
    print("✅ 成功加载.env")
    print("LLM_MODEL =", os.environ.get("LLM_MODEL"))
else:
    print("❌ 没有找到.env文件")

[Errno 2] No such file or directory: 'C:Users21808pyt-aiae202-template'
/mnt/c/Users/21808/pyt-aiae202-template/01-intro-to-ai-agents/code_samples
当前工作目录: /mnt/c/Users/21808/pyt-aiae202-template/01-intro-to-ai-agents/code_samples
找到的.env完整路径: /mnt/c/Users/21808/pyt-aiae202-template/.env
✅ 成功加载.env
LLM_MODEL = deepseek-v4-pro


In [4]:
!pip install jsonpointer

Defaulting to user installation because normal site-packages is not writeable
Could not fetch URL https://pypi.org/simple/jsonpointer/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/jsonpointer/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping
Could not fetch URL https://pypi.org/simple/pip/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pip/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping


ERROR: Could not find a version that satisfies the requirement jsonpointer (from versions: none)
ERROR: No matching distribution found for jsonpointer


In [3]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave*. In LangChain these become the agent's **system prompt**.
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions. The function's **docstring becomes the tool description** the model reads when deciding whether to call it, and its type hints define the arguments.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [4]:
from langchain.tools import tool


@tool
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona", "Paris", "Berlin", "Tokyo", "Sydney",
        "New York City", "Cairo", "Cape Town", "Rio de Janeiro", "Bali",
    ]

Now we wire the model, the tool and the instructions together with `create_agent`. The agent runs a **tool-calling loop**: it sends the conversation to the model, executes any tool the model asks for, feeds the result back, and repeats until the model answers in plain text.

`agent.invoke` returns the full message history — user message, the model's tool call, the tool result, and the final reply. `reply_text` pulls the text out of the last message (some providers return content as a list of blocks rather than a single string).

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[get_destinations],
    system_prompt=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content


result = agent.invoke(
    {"messages": [{"role": "user", "content": "I'm looking for a warm beach destination. What do you recommend?"}]}
)
print(reply_text(result))

Based on your preference for a **warm beach destination**, here are my top recommendations from the available options:

## 🌴 Top Picks

### 1. **Bali, Indonesia**
- **Why:** Tropical paradise with stunning beaches like Nusa Dua, Seminyak, and Uluwatu. Warm year-round, great for surfing, snorkeling, and relaxing.
- **Vibe:** Lush jungle landscapes, beach clubs, wellness retreats, and vibrant culture.

### 2. **Rio de Janeiro, Brazil**
- **Why:** Famous beaches like Copacabana and Ipanema, with warm weather and a lively, energetic atmosphere.
- **Vibe:** Bustling beach culture, samba music, dramatic mountains, and incredible sunsets.

### 3. **Sydney, Australia**
- **Why:** Iconic Bondi and Manly beaches with a warm climate (especially during the Southern Hemisphere summer).
- **Vibe:** Outdoor lifestyle, coastal walks, surfing, and a cosmopolitan city feel.

### 4. **Barcelona, Spain**
- **Why:** Beautiful Mediterranean beaches right in the city, with warm summers and a laid-back seasid

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

`agent.astream(..., stream_mode="messages")` yields `(token, metadata)` pairs for every message chunk the graph produces. We print only the chunks that come from the model node (tool calls and tool results also flow through the stream). Jupyter supports top-level `await`, so the `async for` below runs as-is.

In [6]:
async for token, metadata in agent.astream(
    {"messages": [{"role": "user", "content": "Tell me about Tokyo as a travel destination"}]},
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
        print(token.content, end="", flush=True)
print()

Tokyo is one of the world’s most exciting travel destinations — a huge, safe, and incredibly well-organized city where ultra-modern skyscrapers, neon nightlife, ancient temples, and quiet neighborhood lanes all sit side by side.

### 🗼 Highlights by area
- **Shibuya** – Famous for Shibuya Crossing, shopping, and youthful energy.
- **Shinjuku** – Skyscrapers, nightlife, Golden Gai’s tiny bars, and the peaceful Shinjuku Gyoen Garden.
- **Asakusa** – Home to Senso-ji Temple, Nakamise shopping street, and old-Tokyo atmosphere.
- **Akihabara** – Electronics, anime, manga, and gaming culture.
- **Harajuku / Omotesando** – Trendy fashion, Takeshita Street, and nearby Meiji Shrine.
- **Tokyo Station / Marunouchi** – Great for food, shopping, and access to day trips.
- **Odaiba** – Futuristic entertainment district on Tokyo Bay with malls, museums, and views.

### 🍣 Food
Tokyo has more Michelin-starred restaurants than any other city, but you can also eat incredibly well for cheap. Must-tries i

## Summary

In this lesson you learned how to:

- **Create a model client** with `ChatOpenAI`, which talks to any OpenAI-compatible endpoint — the provider is just configuration.
- **Define a tool** with the `@tool` decorator, which turns a plain Python function (and its docstring) into something the model can call.
- **Create an agent** with `create_agent`, which wires the model, tools and instructions into a tool-calling loop.
- **Stream responses** with `astream` to print tokens as they arrive.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.